# Data Generation

## Fire and Smoke

As a first exercise, we want to generate data from a simple causal diagram that contains a mediator. We will use the following DAG:

```
A -> B -> C
```

Imagine that you want to generate data for a 

```
Fire -> Smoke -> Alarm
```

model. Let fire be a binary variable that indicates whether there is a fire or not, smoke to be a normally distributed variable that indicates the amount of smoke, and alarm to be a binary variable that indicates whether the alarm goes off or not. To simplify things, we assume that the fire alarm activates if the amount of smoke exceeds a certain threshold.

- Write down (on paper) the mathematical model that you would use to generate data for this DAG. You can use the following notation but you should modify it to fit the DAG:

$$
\begin{align*}
\text{Fire} & \sim \text{Bernoulli}(p) \\
\text{Smoke} & \sim \text{Normal}(\mu, \sigma) \\
\text{Alarm} & = \mathbf{1}(\text{Smoke} > \text{threshold})
\end{align*}
$$

- Next, implement the data generation process in `R` in a cell below to create a dataset with 10 observations. You can choose any values for the parameters (p, mu, sigma, threshold) that you think are reasonable. Use the `rbinom` function to generate the binary variable for fire, the `rnorm` function to generate the normally distributed variable for smoke, and a simple logical condition to determine whether the alarm goes off based on the amount of smoke (choose a reasonable threshold).

- Inspect the data you generated to see that it fits what you expected based on the DAG. Then increase the number of observations to 100 and visualize the data (think about what kind of plot would be appropriate and prompt specifically  for it).

![Sir Robert Falcon Scott](https://upload.wikimedia.org/wikipedia/commons/c/cc/Scott_of_the_Antarctic.jpg)

![South Pole Expedition 1911](https://upload.wikimedia.org/wikipedia/commons/thumb/6/6b/Scottgroup.jpg/1920px-Scottgroup.jpg)

In [2]:
if(!requireNamespace("tidyverse", quietly = TRUE)) {
  install.packages("tidyverse")
}

suppressPackageStartupMessages(library(tidyverse))

Warning message:
“package ‘tidyverse’ was built under R version 4.4.3”
Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”


Warning message:
“package ‘tibble’ was built under R version 4.4.3”
Warning message:
“package ‘tidyr’ was built under R version 4.4.3”
Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘purrr’ was built under R version 4.4.3”
Warning message:
“package ‘dplyr’ was built under R version 4.4.3”
Warning message:
“package ‘stringr’ was built under R version 4.4.3”
Warning message:
“package ‘forcats’ was built under R version 4.4.3”
Warning message:
“package ‘lubridate’ was built under R version 4.4.3”


## Gender Discrimination at Berkley College

For homework, see @BICKEL1975SexBiasGraduate.

In [ ]:
suppressPackageStartupMessages(library(tidyverse))

berkley <- read_csv(
  "https://waf.cs.illinois.edu/discovery/berkeley.csv"
  ) %>%
  mutate(
    IsRejected = ifelse(Admission == "Accepted", 0, 1),
    Department = match(tolower(Major), letters),
    Department = replace_na(Department, 0)
  ) |>
  select(-c(Admission, Year, Major))

berkley |> head()

Rows: 12763 Columns: 4
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): Major, Sex, Admission
dbl (1): Year

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Sex,IsRejected,Faculty
<chr>,<dbl>,<int>
F,1,3
M,0,2
F,0,0
M,0,0
M,1,0
M,1,0


In [107]:
berkley |>
    group_by(Sex) |>
    summarize(
        n = n(),
        RejectionRatePct = 100 * mean(IsRejected)
    )

Sex,n,RejectionRatePct
<chr>,<int>,<dbl>
F,4321,65.42467
M,8442,55.72139


In [116]:
rejectionRatesByFaculty <- berkley |>
    group_by(Faculty, Sex) |>
    summarize(
        n = n(),
        RejectionRatePct = round(100 * mean(IsRejected), 1),
    )

rejectionRatesByFaculty

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by Faculty and Sex.
ℹ Output is grouped by Faculty.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(Faculty, Sex))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


Faculty,Sex,n,RejectionRatePct
<int>,<chr>,<int>,<dbl>
0,F,2486,62.3
0,M,5438,59.0
1,F,108,17.6
1,M,1138,27.5
2,F,25,32.0
2,M,560,37.0
3,F,593,66.1
3,M,325,63.1
4,F,375,65.1


In [117]:
rejectionRatesByFaculty |>
    group_by(Faculty) |>
    summarize(
        `Rejection Rate Diff: M - F` = round(diff(RejectionRatePct), 1)
    )

Faculty,Rejection Rate Diff: M - F
<int>,<dbl>
0,-3.3
1,9.9
2,5.0
3,-3.0
4,1.8
5,-3.8
6,1.4
